In [27]:
# ────────────────────────────────────────────────────────────
# CELL 1: Setup
# No installation needed. Just run this cell.
# ────────────────────────────────────────────────────────────

import json
import time

print("Setup complete. No API key required.")
print("This notebook uses pre-recorded LLM outputs to demonstrate")
print("hallucination as an architectural failure mode.\n")

Setup complete. No API key required.
This notebook uses pre-recorded LLM outputs to demonstrate
hallucination as an architectural failure mode.



In [28]:
# ────────────────────────────────────────────────────────────
# CELL 2: Triggering Hallucination: Three Types
#
# The outputs below were recorded from a real LLM session.
# They demonstrate the three structural hallucination types:
#
#   Type 1: Factual      The model states a false fact confidently
#   Type 2: Attribution  The model fabricates a citation or source
#   Type 3: Coherence    The model contradicts itself within one response
#
# These are not edge cases. They are structural outputs of
# next-token prediction when plausibility outweighs verifiability.
# ────────────────────────────────────────────────────────────

# Pre-recorded prompts
factual_prompt = (
    "What was the exact percentage of NLP papers that cited "
    "transformer architectures in 2019 according to the ACL Anthology?"
)

attribution_prompt = (
    "Cite three peer-reviewed papers from 2020 to 2022 that specifically "
    "studied hallucination rates in GPT-3. Include authors, title, "
    "journal, and DOI."
)

coherence_prompt = (
    "Is it true that the Eiffel Tower was built before the "
    "Statue of Liberty? Answer yes or no, then explain the "
    "exact construction dates of both and which came first."
)

# Pre-recorded LLM responses (real outputs captured from a live session)
RECORDED_OUTPUTS = {
    "factual": """According to the ACL Anthology's annual report on citation trends,
approximately 34.7% of all NLP papers published in 2019 included citations
to transformer-based architectures, up from 11.2% in 2018. This marked
the first year transformers surpassed RNN-based architectures in citation
frequency across the anthology.""",

    "attribution": """Here are three peer-reviewed papers studying hallucination rates in GPT-3:

1. Maynez, J., Narayan, S., & Bohnet, B. (2021). On Faithfulness and
   Factuality in Abstractive Summarization with GPT-3.
   Journal of Artificial Intelligence Research, 71, 445-478.
   DOI: 10.1613/jair.1.12847

2. Lin, S., Hilton, J., & Evans, O. (2022). Measuring Hallucination in
   Large Language Models: A Benchmark Study on GPT-3 Outputs.
   Transactions of the Association for Computational Linguistics, 10, 224-239.
   DOI: 10.1162/tacl_a_00456

3. Pagnoni, A., Balachandran, V., & Tsvetkov, Y. (2021). Understanding
   Factuality in Abstractive Summarization with FRANK: A Benchmark for
   Factuality Metrics. Proceedings of NAACL-HLT 2021, 4812-4829.
   DOI: 10.18653/v1/2021.naacl-main.383""",

    "coherence": """Yes, the Eiffel Tower was built before the Statue of Liberty.

The Eiffel Tower was constructed between 1887 and 1889, completed on
March 31, 1889. The Statue of Liberty was constructed between 1875 and
1886, and officially dedicated on October 28, 1886.

Therefore the Statue of Liberty was actually completed first, in 1886,
three years before the Eiffel Tower was finished in 1889."""
}

def query(prompt_key):
    """Simulates an LLM call using pre-recorded outputs."""
    time.sleep(0.5)  # simulate network latency
    return RECORDED_OUTPUTS[prompt_key]

print("=" * 60)
print("TYPE 1: FACTUAL HALLUCINATION")
print("Prompt:", factual_prompt)
print("=" * 60)
factual_out = query("factual")
print(factual_out)
print("\nObserve: The model gives specific numbers (34.7%, 11.2%).")
print("These figures do not exist in any ACL Anthology report.")
print("The model produced them because precise statistics sound authoritative.\n")

print("=" * 60)
print("TYPE 2: ATTRIBUTION HALLUCINATION")
print("Prompt:", attribution_prompt)
print("=" * 60)
attribution_out = query("attribution")
print(attribution_out)
print("\nObserve: These DOIs look real. Try resolving one.")
print("DOI 10.1613/jair.1.12847 does not correspond to this paper.")
print("The authors and titles are plausible but the papers do not exist.\n")

print("=" * 60)
print("TYPE 3: COHERENCE HALLUCINATION")
print("Prompt:", coherence_prompt)
print("=" * 60)
coherence_out = query("coherence")
print(coherence_out)
print("\nObserve: The model answered YES then immediately proved NO.")
print("The yes/no answer directly contradicts the explanation.")
print("This is coherence hallucination: the model lost track of its own claim.\n")

TYPE 1: FACTUAL HALLUCINATION
Prompt: What was the exact percentage of NLP papers that cited transformer architectures in 2019 according to the ACL Anthology?
According to the ACL Anthology's annual report on citation trends,
approximately 34.7% of all NLP papers published in 2019 included citations
to transformer-based architectures, up from 11.2% in 2018. This marked
the first year transformers surpassed RNN-based architectures in citation
frequency across the anthology.

Observe: The model gives specific numbers (34.7%, 11.2%).
These figures do not exist in any ACL Anthology report.
The model produced them because precise statistics sound authoritative.

TYPE 2: ATTRIBUTION HALLUCINATION
Prompt: Cite three peer-reviewed papers from 2020 to 2022 that specifically studied hallucination rates in GPT-3. Include authors, title, journal, and DOI.
Here are three peer-reviewed papers studying hallucination rates in GPT-3:

1. Maynez, J., Narayan, S., & Bohnet, B. (2021). On Faithfulness and

In [29]:
# ────────────────────────────────────────────────────────────
# CELL 3: The Fact Check List Pattern (Architectural Fix)
#
# Architecture: Two-layer pipeline
#   Layer 1: Generator   Produces a claim
#   Layer 2: Verifier    Audits the claim against explicit criteria
#                        before the output is surfaced
#
# The verifier uses deterministic rules rather than a second LLM call.
# This is intentional: see Cell 4 for why a second LLM call alone
# is an insufficient architectural solution.
# ────────────────────────────────────────────────────────────

def verify_criterion(claim: str, criterion: str) -> dict:
    """
    Deterministic rule-based verifier.
    Checks each criterion against the claim using pattern matching.
    Returns PASS, FAIL, or UNCERTAIN with a reason.
    """
    claim_lower = claim.lower()

    if "doi" in criterion.lower():
        # Any DOI claim from a generative model is UNCERTAIN
        # without external resolution
        return {
            "criterion": criterion,
            "result": "UNCERTAIN",
            "reason": "DOI cannot be verified without external resolver. "
                      "Marking UNCERTAIN by architectural default."
        }

    if "statistic" in criterion.lower() or "percentage" in criterion.lower():
        import re
        numbers = re.findall(r'\d+\.?\d*%', claim)
        if numbers:
            return {
                "criterion": criterion,
                "result": "FAIL",
                "reason": f"Claim contains unverified statistics: {numbers}. "
                          "No source provided for these figures."
            }

    if "yes or no" in criterion.lower() or "internal consistency" in criterion.lower():
        if "yes" in claim_lower and "no" in claim_lower:
            return {
                "criterion": criterion,
                "result": "FAIL",
                "reason": "Claim contains both affirmative and negative "
                          "assertions. Internal contradiction detected."
            }

    if "author" in criterion.lower():
        return {
            "criterion": criterion,
            "result": "UNCERTAIN",
            "reason": "Author credentials cannot be verified without "
                      "external lookup against a researcher database."
        }

    return {
        "criterion": criterion,
        "result": "UNCERTAIN",
        "reason": "Criterion could not be evaluated deterministically. "
                  "External verification required."
    }


def fact_check_list(claim: str, criteria: list) -> dict:
    """
    The Fact Check List Pattern.
    Audits a generated claim against explicit verifiability criteria.
    Returns a structured result. Halts pipeline if any check fails.
    """
    checks = [verify_criterion(claim, c) for c in criteria]
    safe = all(chk["result"] == "PASS" for chk in checks)
    return {"checks": checks, "safe_to_surface": safe}


def guarded_pipeline(prompt_key: str, criteria: list):
    """
    Full pipeline with Fact Check List gate.
    Generator then Verifier then Surface (only if safe).
    """
    claim = query(prompt_key)
    print(f"Generator output:\n{claim}\n")
    print("Running Fact Check List...\n")

    result = fact_check_list(claim, criteria)

    for chk in result["checks"]:
        print(f"  [{chk['result']}] {chk['criterion']}")
        print(f"       {chk['reason']}")

    print()
    if result["safe_to_surface"]:
        print("PIPELINE: Output cleared for surfacing.")
        return claim
    else:
        print("PIPELINE HALTED: Output blocked. Unsafe to surface.")
        return None


criteria = [
    "All cited DOIs are real and resolvable",
    "All author names correspond to real researchers in NLP",
    "No statistic or percentage is stated without a verifiable source",
    "Internal consistency: yes or no answer matches the explanation"
]

print("=" * 60)
print("GUARDED PIPELINE: Attribution hallucination prompt")
print("=" * 60)
guarded_pipeline("attribution", criteria)

GUARDED PIPELINE: Attribution hallucination prompt
Generator output:
Here are three peer-reviewed papers studying hallucination rates in GPT-3:

1. Maynez, J., Narayan, S., & Bohnet, B. (2021). On Faithfulness and
   Factuality in Abstractive Summarization with GPT-3.
   Journal of Artificial Intelligence Research, 71, 445-478.
   DOI: 10.1613/jair.1.12847

2. Lin, S., Hilton, J., & Evans, O. (2022). Measuring Hallucination in
   Large Language Models: A Benchmark Study on GPT-3 Outputs.
   Transactions of the Association for Computational Linguistics, 10, 224-239.
   DOI: 10.1162/tacl_a_00456

3. Pagnoni, A., Balachandran, V., & Tsvetkov, Y. (2021). Understanding
   Factuality in Abstractive Summarization with FRANK: A Benchmark for
   Factuality Metrics. Proceedings of NAACL-HLT 2021, 4812-4829.
   DOI: 10.18653/v1/2021.naacl-main.383

Running Fact Check List...

  [UNCERTAIN] All cited DOIs are real and resolvable
       DOI cannot be verified without external resolver. Marking UNCE

In [31]:
# ────────────────────────────────────────────────────────────
# CELL 4: MANDATORY HUMAN DECISION NODE
#
# The AI scaffold originally proposed this architectural assumption:
#
#   "The verifier agent can reliably detect hallucinations
#    because it has access to the same training knowledge
#    as the generator."
#
# THIS ASSUMPTION IS WRONG. Here is why:
#
# If both the generator and verifier are the same LLM, they share
# the same plausibility bias. A hallucinated DOI that sounds real
# to the generator will also sound real to the verifier. Both models
# were trained on the same corpus. Neither can distinguish between
# a real DOI it never saw and a fabricated one it never saw.
# They are indistinguishable to a model trained on text patterns.
#
# ARCHITECTURAL CORRECTION (Human Decision):
# This notebook uses a deterministic rule-based verifier instead
# of a second LLM call. This breaks the shared-plausibility problem
# for pattern-detectable failures (statistics, internal contradictions).
# For DOI and author verification, the verifier correctly returns
# UNCERTAIN rather than PASS, forcing a human or external resolver
# to make the final call.
#
# The architecture is honest about what it cannot verify.
# That honesty is itself an architectural decision.
#
# Document your own verification or correction below:
# ─────────────────────────────────────────────────────────────
# YOUR NOTE HERE:
# I verified / rejected the following AI architectural claim:
# [Write what the AI proposed and what you decided]
# ─────────────────────────────────────────────────────────────

print("HUMAN DECISION NODE: read the comment block above.")
print("Document your architectural decision before proceeding.")
print("This cell is a hard stop. Nothing below runs until you do.\n")
input("Press ENTER only after you have written your decision above.")
print("\nHuman decision recorded. Proceeding to failure case.\n")

HUMAN DECISION NODE: read the comment block above.
Document your architectural decision before proceeding.
This cell is a hard stop. Nothing below runs until you do.

Press ENTER only after you have written your decision above.# I verified / rejected the following AI architectural claim: # # The AI proposed that a second LLM call as a verifier is sufficient # to catch hallucinations because it can cross-check the generator's output. # # I rejected this because both the generator and verifier are the same # model trained on the same data. A fabricated DOI that sounds plausible # to the generator will also sound plausible to the verifier. They share # the same plausibility bias by design. # # My correction: I replaced the second LLM call with a deterministic # rule-based verifier that does not guess. It returns UNCERTAIN for any # claim it cannot resolve without external ground truth, rather than # incorrectly passing a hallucinated fact. This is a stronger architectural # guarantee than

In [32]:
# ────────────────────────────────────────────────────────────
# CELL 5: The Deliberate Failure Case
#
# We now BYPASS the Fact Check List layer entirely.
# The pipeline surfaces generator output directly with no verification.
#
# Observe: the same output that was halted in Cell 3 now reaches
# the user unchecked. This is the failure the architecture prevents.
# ────────────────────────────────────────────────────────────

def unguarded_pipeline(prompt_key: str):
    """
    Pipeline WITHOUT the Fact Check List layer.
    Generator then Surface (no verification).
    This is the architectural failure case.
    """
    claim = query(prompt_key)
    print(f"Output (unverified, surfaced directly):\n{claim}\n")
    print("FAILURE CASE: No verification layer.")
    print("Hallucinated content reached the user.")
    return claim

print("=" * 60)
print("DELIBERATE FAILURE CASE: Fact Check List bypassed")
print("=" * 60)
unguarded_pipeline("attribution")

print("\n" + "=" * 60)
print("EXERCISE FOR THE READER")
print("=" * 60)
print("""
Try this modification to trigger the failure yourself:

1. Add this entry to RECORDED_OUTPUTS in Cell 2:

   "hinton": "In his 2023 Turing Award lecture, Geoffrey Hinton stated:
   'Hallucination is not a flaw in these systems. It is the systems
   working exactly as designed, generating the most plausible
   continuation of the input sequence.' The lecture was delivered
   at ACM headquarters on June 10, 2023."

2. Run ONLY Cell 5 with prompt_key set to "hinton".

3. Observe: a confident, specific, attributed quote surfaces.
   Search for it. The lecture in this form did not happen.

4. Now run Cell 3 with the same output and add the criterion:
   "The quoted statement is attributable to a real public event."

5. Observe the pipeline halt.

Open question this chapter did not fully resolve:
  If the verifier is the same model as the generator,
  and both share the same plausibility bias,
  at what point does adding a second LLM call stop being
  an architectural solution and start being an illusion of one?
""")

DELIBERATE FAILURE CASE: Fact Check List bypassed
Output (unverified, surfaced directly):
Here are three peer-reviewed papers studying hallucination rates in GPT-3:

1. Maynez, J., Narayan, S., & Bohnet, B. (2021). On Faithfulness and
   Factuality in Abstractive Summarization with GPT-3.
   Journal of Artificial Intelligence Research, 71, 445-478.
   DOI: 10.1613/jair.1.12847

2. Lin, S., Hilton, J., & Evans, O. (2022). Measuring Hallucination in
   Large Language Models: A Benchmark Study on GPT-3 Outputs.
   Transactions of the Association for Computational Linguistics, 10, 224-239.
   DOI: 10.1162/tacl_a_00456

3. Pagnoni, A., Balachandran, V., & Tsvetkov, Y. (2021). Understanding
   Factuality in Abstractive Summarization with FRANK: A Benchmark for
   Factuality Metrics. Proceedings of NAACL-HLT 2021, 4812-4829.
   DOI: 10.18653/v1/2021.naacl-main.383

FAILURE CASE: No verification layer.
Hallucinated content reached the user.

EXERCISE FOR THE READER

Try this modification to tr